# Laboratory 6 — RGB, Vibration, and LiDAR Encoding

Companion to Chapters 5–8. All input data are generated here. No camera, IMU, LiDAR, ROS, or Gazebo is required. The examples test numerical mappings, not real-world detector accuracy.

Run all cells, then change one parameter at a time. Figures are synthetic teaching outputs.

[Run in Colab](https://colab.research.google.com/github/an-hongyu/Introduction-to-Neuromorphic-Embodied-AI/blob/main/notebooks/sensory_encoding.ipynb)

## 1. Physical-unit LIF helper

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simulate_lif(current_pa, dt_ms=0.1, capacitance_pf=200.0,
                 leak_ns=10.0, rest_mv=-65.0, threshold_mv=-50.0,
                 reset_mv=-65.0, refractory_ms=2.0, spiking=True):
    """One current value per time interval; return time, voltage, spike times.

    Units: pA, ms, pF, nS, mV. Initial voltage is reset_mv.
    Exact passive update for piecewise-constant current; endpoint events.
    spiking=False exposes the passive RC response without threshold/reset.
    """
    current = np.asarray(current_pa, dtype=float)
    if current.ndim != 1 or current.size == 0 or not np.isfinite(current).all():
        raise ValueError('Supply a nonempty, finite 1-D current array.')
    if dt_ms <= 0 or capacitance_pf <= 0 or leak_ns <= 0 or refractory_ms < 0:
        raise ValueError('dt, capacitance, and leak must be positive; refractory >= 0.')
    if reset_mv >= threshold_mv:
        raise ValueError('Reset must be below threshold.')
    tau_ms = capacitance_pf / leak_ns
    decay = np.exp(-dt_ms / tau_ms)
    time = np.arange(current.size + 1) * dt_ms
    voltage = np.empty(time.size)
    voltage[0] = reset_mv
    spikes, hold = [], 0
    # Round refractory duration upward to a whole number of time steps.
    refractory_steps = int(np.ceil(refractory_ms / dt_ms - 1e-12))
    for k, injected_pa in enumerate(current):
        if hold > 0:
            voltage[k + 1] = reset_mv
            hold -= 1
        else:
            steady_mv = rest_mv + injected_pa / leak_ns
            voltage[k + 1] = steady_mv + (voltage[k] - steady_mv) * decay
            if spiking and voltage[k + 1] >= threshold_mv:
                spikes.append(time[k + 1])
                voltage[k + 1] = reset_mv
                hold = refractory_steps
    return time, voltage, np.asarray(spikes)

def pulse_train(time_ms, starts_ms, width_ms=4.0, amplitude_pa=600.0):
    """Rectangular current pulses; time_ms contains interval start times."""
    current = np.zeros_like(time_ms, dtype=float)
    for start in starts_ms:
        current += amplitude_pa * ((time_ms >= start) & (time_ms < start + width_ms))
    return current

def analytical_rate(current_pa, capacitance_pf=200.0, leak_ns=10.0,
                    rest_mv=-65.0, threshold_mv=-50.0, reset_mv=-65.0,
                    refractory_ms=2.0):
    steady_mv = rest_mv + current_pa / leak_ns
    if steady_mv <= threshold_mv:
        return 0.0
    crossing_ms = (capacitance_pf / leak_ns) * np.log(
        (steady_mv - reset_mv) / (steady_mv - threshold_mv))
    return 1000.0 / (crossing_ms + refractory_ms)

def plot_response(ax, time, voltage, spikes, label, color, threshold=-50.0):
    ax.plot(time, voltage, color=color, label=label, lw=1.8)
    if len(spikes):
        ax.vlines(spikes, threshold + 0.7, threshold + 3.0, color=color, lw=1.5)
    ax.axhline(threshold, color='0.55', ls='--', lw=0.9)
    ax.set(xlabel='Time (ms)', ylabel='Voltage (mV)')

## 2. Feature definitions and validity masks

In [ ]:
def red_feature(rgb):
    rgb = np.asarray(rgb, dtype=float)
    if rgb.ndim < 1 or rgb.shape[-1] != 3 or not np.isfinite(rgb).all() or np.any((rgb < 0) | (rgb > 1)):
        raise ValueError('RGB values must be normalized into [0,1].')
    return np.clip(rgb[...,0] - .5*(rgb[...,1]+rgb[...,2]), 0, 1).mean()

def causal_rms(signal, window):
    return np.sqrt(np.convolve(np.asarray(signal)**2, np.ones(window)/window, mode='valid'))

def sector_encode(ranges, angles, range_min=.05, range_max=5., sectors=36):
    ranges, angles = np.asarray(ranges), np.asarray(angles)
    good = np.isfinite(ranges) & np.isfinite(angles) & (ranges >= range_min) & (ranges <= range_max)
    safe_angles = np.where(np.isfinite(angles), angles, 0.)
    bins = np.floor(((safe_angles + np.pi) % (2*np.pi)) / (2*np.pi) * sectors).astype(int)
    distance = np.full(sectors, np.nan)
    for k in range(sectors):
        values = ranges[good & (bins == k)]
        if len(values): distance[k] = np.min(values)
    valid = np.isfinite(distance)
    rates = np.zeros(sectors)
    rates[valid] = 5 + 45*(1 - distance[valid]/range_max)
    return distance, rates, valid

## 3. Synthetic color patches

In [ ]:
patches = {'Red': [.9,.1,.1], 'White': [.8,.8,.8], 'Blue': [.1,.1,.9], 'Dim red': [.45,.05,.05]}
features = [red_feature(np.tile(rgb, (12,12,1))) for rgb in patches.values()]
counts = [len(simulate_lif(np.full(5000, 50+250*x))[2]) for x in features]
fig, ax = plt.subplots(1,2,figsize=(10,4),constrained_layout=True)
ax[0].bar(list(patches), features, color=['#b75d45','.7','#24628a','#ce998c']); ax[0].set(ylabel='Red-excess feature [0,1]',title='A. Synthetic RGB patches')
ax[1].bar(list(patches), counts, color='#24628a'); ax[1].set(ylabel='LIF spikes / 500 ms',title='B. Declared mapping: I = 50 + 250s pA')
plt.show()
assert features[0] > features[3] > features[1] == features[2] == 0
assert counts[0] > 0 and counts[1] == counts[2] == 0

## 4. Causal vibration detection

In [ ]:
rng = np.random.default_rng(2026)
time = np.arange(600) / 200
active = (time >= .8) & (time < 1.6)
raw = 9.81 + active*np.sin(2*np.pi*20*time) + .02*rng.standard_normal(len(time))
bias = raw[time < .3].mean()  # stationary, fixed-orientation calibration only
window = 20
rms = causal_rms(raw-bias, window)
tr = time[window-1:]
fig, ax = plt.subplots(2,1,figsize=(10,5),sharex=True,constrained_layout=True)
ax[0].plot(time,raw,color='#24628a'); ax[0].set(ylabel='Sensor z (m/s²)',title='A. Synthetic acceleration with stationary gravity offset')
ax[1].plot(tr,rms,color='#24628a'); ax[1].axhline(.4,ls='--',color='#b75d45',label='Chosen detector threshold')
ax[1].set(xlabel='Time (s)',ylabel='Trailing RMS (m/s²)',title='B. Causal 100 ms window'); ax[1].legend()
plt.show()
assert np.median(rms[(tr>1)&(tr<1.4)]) > .6
assert np.median(rms[tr<.6]) < .05

## 5. Thirty-six LiDAR sectors

In [ ]:
angles = np.linspace(-np.pi,np.pi,360,endpoint=False)
ranges = np.full(360,3.0); ranges[np.abs(angles)<np.deg2rad(15)] = .8
ranges[100:110] = np.nan  # an entire sector is missing, not free space
distance, rates, valid = sector_encode(ranges,angles)
centers = np.deg2rad(np.arange(36)*10-175)
fig, ax = plt.subplots(1,2,figsize=(10,4),constrained_layout=True)
ax[0].plot(np.rad2deg(centers),distance,'o-',color='#24628a'); ax[0].set(xlabel='Body-relative sector center (deg)',ylabel='Nearest valid range (m)',title='A. 36 sectors, 10° each')
ax[1].bar(np.rad2deg(centers),rates,width=8,color=np.where(valid,'#24628a','#b75d45')); ax[1].set(xlabel='Sector center (deg)',ylabel='Prescribed rate (Hz)',title='B. Unknown sector: zero input + invalid mask')
plt.show()
assert len(distance)==36 and (~valid).any()
assert rates[valid & (distance<1)].min() > rates[valid & (distance>2)].max()
assert np.all(rates[~valid]==0)
print('Sensor checks passed: RGB mapping, causal RMS, 36-sector mask and monotone range encoding.')

## References

[1] P. Corke, *Robotics, Vision and Control: Fundamental Algorithms in Python*. Springer, 2023. [DOI](https://doi.org/10.1007/978-3-031-06469-2).

[2] ROS 2 contributors, “LaserScan message.” [Message contract](https://github.com/ros2/common_interfaces/blob/jazzy/sensor_msgs/msg/LaserScan.msg). All feature functions and numerical examples in this notebook are original teaching choices.